In [3]:
# --- imports ---
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from ib_insync import Stock

sys.path.append(os.path.abspath(".."))
from ibkr.Class_IBKR_IB import IBKR_IB


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# Use ["long"], ["short"], or ["long", "short"].
# ============================================================
sorted_symbols_list = ["CWB", "ICVT"]
trade_directions = ["long", "short"]
ibkr_port = 7496
lookback_period = "5 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
prices_to_use = "TRADES"
moving_avg_days = 20
start_date = pd.Timestamp("2024-01-02")
end_date = pd.Timestamp("2024-12-15")
hurdle_step = 0.001
number_of_entry_steps = 10
commission_per_share = 0.005
dollar_constant = 100_000
trading_days_per_year = 252
progress_interval = 10
output_directory = Path("backtests")
summary_output_directory = Path("summary_stats")


def report(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


def enhance_prices(closes_df):
    df = closes_df.copy()
    df.reset_index(names="to date", inplace=True)
    df.insert(0, "from date", df["to date"].shift(1))

    for symbol in sorted_symbols_list:
        from_price = f"from {symbol} price"
        to_price = f"to {symbol} price"
        df[from_price] = df[symbol].shift(1)
        df[to_price] = df[symbol]
        df[f"{symbol} cod"] = df[to_price] - df[from_price]
        df[f"{symbol} pct cod"] = np.log(df[to_price] / df[from_price])
        df.drop(columns=symbol, inplace=True)

    anchor = sorted_symbols_list[-1]
    for symbol in sorted_symbols_list:
        df[f"to {anchor} / to {symbol}"] = (
            df[f"to {anchor} price"] / df[f"to {symbol} price"]
        )

    df["from date"] = pd.to_datetime(df["from date"], errors="coerce")
    df["to date"] = pd.to_datetime(df["to date"], errors="coerce")
    date_mask = (
        df["from date"].between(start_date, end_date)
        & df["to date"].between(start_date, end_date)
    )
    df = df.loc[date_mask].reset_index(drop=True)

    return df


def calculate_unit_prices(enhanced_df):
    df = enhanced_df.copy()
    anchor = sorted_symbols_list[-1]

    for symbol in sorted_symbols_list:
        ratio_column = f"to {anchor} / to {symbol}"
        average_column = f"{anchor} / {symbol} moving avg"
        prior_average_column = f"prev {average_column}"
        df[average_column] = df[ratio_column].rolling(moving_avg_days).mean()
        df[prior_average_column] = df[average_column].shift(1)
        df[f"to {symbol} unit price"] = (
            df[prior_average_column] * df[f"to {symbol} price"]
        )

    for symbol in sorted_symbols_list:
        df[f"{symbol} unit price pct diff"] = np.log(
            df[f"to {symbol} unit price"]
            / df[f"to {anchor} unit price"]
        )

    non_anchor = sorted_symbols_list[0]
    df["unit price pct diff"] = (
        df[f"{non_anchor} unit price pct diff"]
        - df[f"{anchor} unit price pct diff"]
    )
    return df


def build_entry_hurdles(direction):
    sign = -1 if direction == "long" else 1
    return [0.0] + [
        round(sign * hurdle_step * step, 4)
        for step in range(1, number_of_entry_steps + 1)
    ]


def build_exit_hurdles(direction, entry_hurdle):
    sign = -1 if direction == "long" else 1
    number_of_steps = int(round(abs(entry_hurdle) / hurdle_step))
    return [0.0] + [
        round(sign * hurdle_step * step, 4)
        for step in range(1, number_of_steps + 1)
    ]


def add_positions(base_df, direction, entry_hurdle, exit_hurdle):
    df = base_df.copy()
    entry_column = f"go {direction}"
    exit_column = f"exit {direction}"

    if direction == "long":
        df[entry_column] = df["unit price pct diff"] < entry_hurdle
        df[exit_column] = df["unit price pct diff"] > exit_hurdle
        position_value = 1
    else:
        df[entry_column] = df["unit price pct diff"] > entry_hurdle
        df[exit_column] = df["unit price pct diff"] < exit_hurdle
        position_value = -1

    df["current_position"] = 0
    df["new_position"] = 0
    next_trading_date = df["to date"].shift(-1)
    df["month end liquidation"] = (
        next_trading_date.notna()
        & (df["to date"].dt.to_period("M") != next_trading_date.dt.to_period("M"))
    )
    current_column = df.columns.get_loc("current_position")
    new_column = df.columns.get_loc("new_position")

    for row_index in range(moving_avg_days, len(df)):
        current_position = df.iat[row_index - 1, new_column]
        new_position = current_position
        if df["month end liquidation"].iat[row_index]:
            new_position = 0
        elif current_position == 0 and df[entry_column].iat[row_index]:
            new_position = position_value
        elif current_position == position_value and df[exit_column].iat[row_index]:
            new_position = 0
        df.iat[row_index, current_column] = current_position
        df.iat[row_index, new_column] = new_position

    return df


def add_shares(position_df):
    df = position_df.copy()

    non_anchor = sorted_symbols_list[0]
    non_anchor_shares = f"{non_anchor} shares per unit"
    df[non_anchor_shares] = 0
    
    anchor = sorted_symbols_list[-1]
    anchor_shares = f"{anchor} shares per unit"
    anchor_average = f"prev {anchor} / {anchor} moving avg"
    df[anchor_shares] = (
        df[anchor_average] * dollar_constant / df[f"to {anchor} price"]
    ).round()

    prior_average = f"prev {anchor} / {non_anchor} moving avg"
    df[non_anchor_shares] = (df[anchor_shares] * df[prior_average]).round()

    for symbol in sorted_symbols_list:
        shares_per_unit = f"{symbol} shares per unit"

        leg_sign = 1 if symbol == non_anchor else -1

        current_shares = f"{symbol} current shares"
        df[current_shares] = None

        target_shares = f"{symbol} target shares"
        df[target_shares] = df["new_position"] * leg_sign * df[shares_per_unit]

        df[current_shares] = df[target_shares].shift(1)
        df.iat[moving_avg_days, df.columns.get_loc(current_shares)] = 0

        df[f"{symbol} shares to trade"] = df[target_shares] - df[current_shares]
        df[f"{symbol} shares to buy"] = (
            df[f"{symbol} shares to trade"].clip(lower=0)
        )
        df[f"{symbol} shares to sell"] = (
            -df[f"{symbol} shares to trade"].clip(upper=0)
        )

    return df
        

def add_stats(shares_df):
    df = shares_df.copy()
    for symbol in sorted_symbols_list:
        current_shares = f"{symbol} current shares"
        df[f"{symbol} daily position pnl"] = (
                df[current_shares] * df[f"{symbol} cod"]
            )
        df[f"{symbol} daily commission"] = (
                -df[f"{symbol} shares to trade"].abs() * commission_per_share
            )
        df[f"{symbol} investment amount"] = (
            df[current_shares] * df[f"from {symbol} price"]
        )
        

    profit_columns = [f"{s} daily position pnl" for s in sorted_symbols_list]
    commission_columns = [f"{s} daily commission" for s in sorted_symbols_list]
    investment_columns = [f"{s} investment amount" for s in sorted_symbols_list]
    df["daily position pnl"] = df[profit_columns].sum(axis=1)
    df["daily total commissions"] = df[commission_columns].sum(axis=1)
    df["daily net profit"] = df["daily position pnl"] + df["daily total commissions"]
    df["cumulative net profit"] = df["daily net profit"].cumsum()
    df["drawdown"] = (
        df["cumulative net profit"] - df["cumulative net profit"].cummax()
    )

    df["gross investment amount"] = df[investment_columns].abs().sum(axis=1)
    df["net investment amount"] = df[investment_columns].sum(axis=1)
    return df


def add_summary_stats(stats_df, direction, entry_hurdle, exit_hurdle):
    df = stats_df
    traded_columns = [f"{s} shares to trade" for s in sorted_symbols_list]
    valid_rows = df.index >= moving_avg_days
    daily_profit = df.loc[valid_rows, "daily net profit"]
    average_investment = df.loc[valid_rows, "gross investment amount"].mean()
    total_profit = daily_profit.sum()
    number_of_days = daily_profit.notna().sum()
    annualized_return = (
        total_profit / average_investment * trading_days_per_year / number_of_days
        if average_investment > 0 and number_of_days > 0
        else np.nan
    )
    daily_std = daily_profit.std()
    annualized_sharpe = (
        daily_profit.mean() / daily_std * np.sqrt(trading_days_per_year)
        if daily_std != 0 and not pd.isna(daily_std)
        else np.nan
    )
    return pd.DataFrame(
        [{
            "symbols": "_".join(sorted_symbols_list),
            "moving avg days": moving_avg_days,
            "direction": direction,
            "entry hurdle": entry_hurdle,
            "exit hurdle": exit_hurdle,
            "total net profit": total_profit,
            "average daily investment": average_investment,
            "annualized return on avg investment": annualized_return,
            "annualized Sharpe": annualized_sharpe,
            "maximum drawdown": df["drawdown"].min(),
            "position changes": df["new_position"].ne(df["current_position"]).sum(),
            "total shares traded": df[traded_columns].abs().sum().sum(),
        }]
    )


async def main():
    valid_directions = {"long", "short"}
    if len(sorted_symbols_list) != 2:
        raise ValueError("This pipeline requires exactly two symbols")
    if set(trade_directions) - valid_directions:
        raise ValueError("trade_directions may contain only 'long' and 'short'")
    if hurdle_step <= 0:
        raise ValueError("hurdle_step must be greater than zero")

    report("Starting full backtest pipeline")
    ibkr = IBKR_IB(port=ibkr_port)
    report(f"Connecting to IBKR on port {ibkr_port}")
    await ibkr.connect()
    report("IBKR connection established")

    try:
        contracts = []
        for symbol in sorted_symbols_list:
            report(f"Qualifying {symbol}")
            contract = Stock(symbol, "SMART", "USD")
            await ibkr.ib.qualifyContractsAsync(contract)
            contracts.append(contract)

        report(f"Downloading {lookback_period} of historical prices")
        closes_df = await ibkr.get_historical_closes_df(
            contracts,
            lookback_period=lookback_period,
            length_of_each_period=length_of_each_period,
            prices_to_use=prices_to_use,
            use_regular_trading_hours=use_regular_trading_hours,
            remove_last_row=True,
        )
        report(f"Downloaded {len(closes_df):,} completed price rows")

        enhanced_df = enhance_prices(closes_df)
        report(
            f"Enhanced-price calculations and date filtering complete: "
            f"{len(enhanced_df):,} rows remain"
        )
        unit_price_df = calculate_unit_prices(enhanced_df)
        report("Unit-price calculations complete")
        position_base_df = unit_price_df

        strategy_count = sum(
            len(build_exit_hurdles(direction, entry))
            for direction in trade_directions
            for entry in build_entry_hurdles(direction)
        )
        report(f"Calculating {strategy_count} position/statistics combinations")

        completed_results = []
        summary_dfs = []
        completed_count = 0
        filename_root = f"{'_'.join(sorted_symbols_list)}_{moving_avg_days}"
        for direction in trade_directions:
            report(f"Starting {direction} strategies")
            for entry_hurdle in build_entry_hurdles(direction):
                for exit_hurdle in build_exit_hurdles(direction, entry_hurdle):
                    position_df = add_positions(
                        position_base_df, direction, entry_hurdle, exit_hurdle
                    )
                    shares_df = add_shares(position_df)
                    final_df = add_stats(shares_df)
                    summary_dfs.append(
                        add_summary_stats(
                            final_df, direction, entry_hurdle, exit_hurdle
                        )
                    )
                    filename = (
                        f"{filename_root}_{direction}_"
                        f"{entry_hurdle:.4f}_{exit_hurdle:.4f}.csv"
                    )
                    completed_results.append((filename, final_df))
                    completed_count += 1
                    if (
                        completed_count % progress_interval == 0
                        or completed_count == strategy_count
                    ):
                        report(
                            f"Calculated {completed_count}/{strategy_count} strategies"
                        )

        report("All calculations complete; beginning final save")
        output_directory.mkdir(parents=True, exist_ok=True)
        for save_count, (filename, final_df) in enumerate(completed_results, start=1):
            final_df.to_csv(output_directory / filename)
            if save_count % progress_interval == 0 or save_count == strategy_count:
                report(f"Saved {save_count}/{strategy_count} final files")

        summary_output_directory.mkdir(parents=True, exist_ok=True)
        summary_df = pd.concat(summary_dfs, ignore_index=True)
        summary_filename = f"{filename_root}_summary.csv"
        summary_df.to_csv(summary_output_directory / summary_filename, index=False)
        report(f"Saved consolidated summary: {summary_filename}")

        report(
            f"Pipeline finished successfully: {strategy_count} detail files "
            f"and 1 summary file saved"
        )
    finally:
        ibkr.ib.disconnect()
        report("IBKR disconnected")


await main()


[22:01:53] Starting full backtest pipeline
[22:01:53] Connecting to IBKR on port 7496
[22:01:53] IBKR connection established
[22:01:53] Qualifying CWB
[22:01:53] Qualifying ICVT
[22:01:53] Downloading 5 Y of historical prices
[22:01:54] Downloaded 1,252 completed price rows
[22:01:54] Enhanced-price calculations and date filtering complete: 240 rows remain
[22:01:54] Unit-price calculations complete
[22:01:54] Calculating 132 position/statistics combinations
[22:01:54] Starting long strategies
[22:01:54] Calculated 10/132 strategies
[22:01:54] Calculated 20/132 strategies
[22:01:55] Calculated 30/132 strategies
[22:01:55] Calculated 40/132 strategies
[22:01:55] Calculated 50/132 strategies
[22:01:56] Calculated 60/132 strategies
[22:01:56] Starting short strategies
[22:01:56] Calculated 70/132 strategies
[22:01:57] Calculated 80/132 strategies
[22:01:57] Calculated 90/132 strategies
[22:01:57] Calculated 100/132 strategies
[22:01:58] Calculated 110/132 strategies
[22:01:58] Calculated 